# 16 · Sparse Retrieval：BM25 / 倒排索引

> 稠密检索懂“语义”但可能漏掉精确词（专有名词、型号）。稀疏检索做词项精确匹配，两者互补。

**本文件覆盖知识点**：BM25 / TF-IDF / Inverted Index / Keyword Search；Dense+Sparse 为何更好

```text
"Redis 单线程" → 分词 [redis, 单线程] → 倒排索引定位含这些词的文档
```

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 从 TF-IDF 到 BM25

- **TF-IDF**：词频(TF)越高越重要；但出现在越多文档的词(IDF)越不区分文档 → 降权；
- **BM25**：TF-IDF 的进阶，引入词频饱和（词出现 100 次不等于 100 倍重要）与文档长度归一，是当前关键词检索的事实标准。

倒排索引（Inverted Index）是它们的底座：
```text
词          →  出现该词的文档列表
"redis"    →   [doc3, doc17]
"单线程"    →   [doc3]
```
查询时把词的倒排表取交集/并集打分，无需全表扫描。

In [ ]:
# 用 rank_bm25 在示例库上做关键词检索
from pathlib import Path
from rank_bm25 import BM25Okapi
import numpy as np

def tokenize(text):
    """中文简分：字符 unigram + bigram（演示用；生产换 jieba）"""
    t = ''.join(text.split())
    return list(t) + [t[i:i+2] for i in range(len(t)-1)]

docs = [p.read_text(encoding='utf-8') for p in sorted(Path('data').glob('*.md'))
        if not p.name.startswith('评测集')]   # 人工标注的“标准答案”不进检索库，与主干检索课口径一致
bm = BM25Okapi([tokenize(d) for d in docs])

q = '私有化部署'
scores = bm.get_scores(tokenize(q))
for i in np.argsort(-scores)[:3]:
    print(f'{scores[i]:.2f}  {docs[i][:40]}...')

## 2. BM25 的好处与局限

- 好处：精确词命中、零成本、离线可调、可解释（知道命中了哪个词），对专有名词和型号非常可靠；
- 局限：不懂同义改写，“咋收费啊”匹配不到“价格档位”；
- Dense 恰好相反：懂同义，但可能对精确型号拿不准。

> 结论：Dense + Sparse 混合通常比单独用任何一种都好（下一课做融合）。

In [ ]:
# 知识点·真调说明：同义改写与词形局限 —— 让模型做“语义等价判断”，反衬 BM25 纯词形匹配会漏同义表达
_llm_live(
    prompt='请逐组判断下面每一组里的两句话是否表达同一个检索意图，并各用一句话说明依据：\n'
           '第1组：A“咋收费啊” / B“价格档位与套餐费用是多少”\n'
           '第2组：A“改价” / B“调整售价”\n'
           '第3组：A“苹果降价了买两斤” / B“苹果新款手机值得买吗”\n'
           '最后单列一行回答：若检索只做“字面词形匹配”（BM25/倒排），上面哪几组会被漏召回，哪一组反而会“假命中”，为什么。',
    system='你是搜索相关性评审。要求逐组判定“同意图/不同意图”，每组一句话理由，最后给一行“词形匹配会漏掉/假命中”的结论。',
    fallback='未配置 Key 的固定样例：\n'
             '第1组：同意图。“咋收费”是口语，“价格档位与套餐费用”是书面说法，语义指向同一件事。\n'
             '第2组：同意图。“改价”与“调整售价”是同义动词，口语/书面之别。\n'
             '第3组：不同意图。A 说的是水果，B 说的是手机品牌，只是字面都含“苹果”。\n'
             '词形匹配结论：第1、2 组几乎没有共享的字/词，会被漏召回；第3 组因共享“苹果”反而假命中——'
             '它既漏同义、又无法对同形词消歧。',
    temperature=0.2,
)
print('→ 这就是“稀疏检索不懂同义改写”的来源；上面 rank_bm25 能精确命中“私有化部署”，'
      '换成口语同义表达就会漏——语义补充正是稠密检索的本课价值。')

## 3. 为什么混合更好

```text
查"SKU-238 怎么改价"
  Sparse: 精确命中 SKU-238 的文档   ← 稠密可能搜不准型号
  Dense:  命中"改价/价格操作"语义   ← 稀疏可能不懂"改价"≈"调整售价"
  → 合并两边结果 = 又准又全
```



In [ ]:
# 知识点·真调说明：混合分工 —— 让模型把一条“带型号的口语问题”拆成给稀疏与稠密两侧的查询
_llm_live(
    prompt='用户问题：“SKU-238 还能改价吗？我是年付的老客户，能退差价吗？”\n'
           '一个混合检索系统要把这个问题喂给两条通道：稀疏检索(BM25，认字面/型号) 和稠密检索(向量，认语义)。\n'
           '请给出两部分：① 哪些词应“原样保留”给稀疏侧（不能改写/换词），为什么；'
           '② 把整句改写成一段适合稠密语义检索的查询（保留原意图）。',
    system='你是检索查询理解工程师。要求只输出①②两段，语言精炼，不写代码。',
    fallback='未配置 Key 的固定样例：\n'
             '① 原样保留给稀疏侧：“SKU-238”（精确型号，改写即失效），另加“改价”“退差价”做词项命中。'
             '原因：稀疏侧的优势就是精确词，型号/专名一旦被同义改写就召回不到对应文档。\n'
             '② 稠密侧改写：查询“SKU-238 型号商品的改价与年付老客户退差价政策”——保留型号以便向量对齐，'
             '同时把“改价/退差价”这类意图扩成语义相近的表达，让稠密召回能命中讲“价格调整/退款规则”的段落。',
    temperature=0.2,
)
print('→ 拆给两侧不是“二选一”：精确词交给稀疏、语义交给稠密，再合并排序——这正是下一课 Hybrid Search 的入口。')

## 小结

- 稀疏检索（BM25/倒排）擅长精确词，稠密擅长语义；
- 中文要分词（生产用 jieba 等）；
- 两者融合是生产标配 → 下一课 Hybrid Search。